# Zone Cutter
Interactive notebook for selecting an XRF spectral dataset, inspecting its metadata, and visualizing raw vs preprocessed spectra.

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

pd.options.plotting.backend = 'plotly'

# Resolve workspace root (experiments/ is one level below root)
workspace_root = Path.cwd().parent.resolve()
smx_dir = workspace_root / 'smx'
if str(smx_dir) not in sys.path:
    sys.path.insert(0, str(smx_dir))

import preprocessings as prepr
import kennard_stone as ks

datasets_dir = workspace_root / 'real_datasets' / 'xrf'
csv_files = sorted([f.stem for f in datasets_dir.glob('*.csv')])

In [ ]:
# Shared state populated when a dataset is selected
Xcalclass = None
Xcalclass_prep = None

def load_dataset(dataset_name):
    global Xcalclass, Xcalclass_prep
    csv_path = datasets_dir / f'{dataset_name}.csv'
    json_path = datasets_dir / f'{dataset_name}.json'

    data_complete = pd.read_csv(csv_path, sep=';')

    # Determine spectral columns from JSON if available, else use all numeric columns
    if json_path.exists():
        with open(json_path) as f:
            meta = json.load(f)
        start_col, end_col = meta['spectral_range']
        class_col = meta.get('class_column', 'Class')
    else:
        numeric_cols = data_complete.select_dtypes(include='number').columns.tolist()
        start_col, end_col = numeric_cols[0], numeric_cols[-1]
        class_col = 'Class'

    classes = data_complete[class_col].unique()
    cal_parts, pred_parts, y_cal_parts = [], [], []

    for cls in classes:
        subset = data_complete[data_complete[class_col] == cls].reset_index(drop=True)
        X_cls = subset.loc[:, start_col:end_col]
        X_cal, X_pred = ks.train_test_split(X_cls, test_size=0.30)
        cal_parts.append(X_cal.reset_index(drop=True))
        pred_parts.append(X_pred.reset_index(drop=True))
        y_cal_parts.extend([cls] * len(X_cal))

    Xcalclass = pd.concat(cal_parts, axis=0).reset_index(drop=True)

    Xcalclass_prep, mean_cal, mean_cal_poisson = prepr.poisson(Xcalclass, mc=True)
    Xcalclass_prep.columns = Xcalclass.columns

    return Xcalclass, Xcalclass_prep

output_load = widgets.Output()

dataset_dropdown = widgets.Dropdown(
    options=csv_files,
    description='Dataset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

def on_dataset_change(change):
    global Xcalclass, Xcalclass_prep
    with output_load:
        clear_output()
        Xcalclass, Xcalclass_prep = load_dataset(change['new'])
        print(f"Loaded '{change['new']}' — calibration set: {Xcalclass.shape}")

dataset_dropdown.observe(on_dataset_change, names='value')

# Load default on startup
Xcalclass, Xcalclass_prep = load_dataset(dataset_dropdown.value)
print(f"Loaded '{dataset_dropdown.value}' — calibration set: {Xcalclass.shape}")

display(dataset_dropdown, output_load)

In [ ]:
# Display JSON metadata for the selected dataset
selected = dataset_dropdown.value
json_path = datasets_dir / f'{selected}.json'

if json_path.exists():
    with open(json_path) as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2, ensure_ascii=False))
else:
    print(f"No JSON metadata file found for dataset: '{selected}'")

In [ ]:
import plotly.graph_objects as go

# Load spectral cuts from JSON metadata if available
selected = dataset_dropdown.value
json_path = datasets_dir / f'{selected}.json'
spectral_cuts = []
if json_path.exists():
    with open(json_path) as f:
        _meta = json.load(f)
    spectral_cuts = _meta.get('spectral_cuts', [])

def add_spectral_cuts(fig, spectral_cuts):
    """Add vertical lines and zone labels for each spectral cut."""
    boundaries = set()
    for zone_name, start, end in spectral_cuts:
        boundaries.add(start)
        boundaries.add(end)

    for b in sorted(boundaries):
        fig.add_vline(
            x=b,
            line=dict(color='rgba(100,100,100,0.4)', width=1, dash='dot')
        )

    # Add zone name annotations at the midpoint of each zone (top of chart)
    for zone_name, start, end in spectral_cuts:
        mid = (start + end) / 2
        fig.add_annotation(
            x=mid, y=1, yref='paper',
            text=zone_name,
            showarrow=False,
            textangle=-90,
            font=dict(size=8, color='gray'),
            yanchor='top',
            xanchor='center'
        )

def plot_spectra(df, title):
    """Plot transposed spectra with a numeric x-axis."""
    T = df.T.copy()
    T.index = pd.to_numeric(T.index, errors='coerce')
    return T.plot(title=title)

# Plot raw and preprocessed calibration spectra (transposed)
fig_raw = plot_spectra(Xcalclass, 'Raw Spectra')
fig_raw.update_layout(showlegend=False, xaxis_title='Energy', yaxis_title='Intensity')
add_spectral_cuts(fig_raw, spectral_cuts)
fig_raw.show()

fig_prep = plot_spectra(Xcalclass_prep, 'Preprocessed Spectra')
fig_prep.update_layout(showlegend=False, xaxis_title='Energy', yaxis_title='Intensity')
add_spectral_cuts(fig_prep, spectral_cuts)
fig_prep.show()
